# 02 · Generate the simulated store data and check it against the real statistics

**Why simulated?** No real, recent UK customer-level transaction data is public: retailers do not publish it and GDPR protects it.
So the transactions belong to a **fictional retailer, Hearthline**. The amounts are **calibrated to real official data** from notebook 01:

| What is simulated | Calibrated to (real) |
|---|---|
| Each store's share of trade | City retail output (ONS GVA, 2024) |
| Items per basket | Household income per head (ONS GDHI 2024 / CSO) |
| Share of orders placed online, by month | ONS internet share of retail (J4MC), Jan 2024 – Sep 2026 |
| Where online customers live | City spending power (population × income) |
| Monthly seasonality and annual growth | ONS average weekly retail sales (JE4W), 2019–2025 |
| Customers' age bands | ONS mid-2025 population estimates (adults) |
| Base price of each product type | Median price on Dunelm's website category pages, Oct 2026 (industry benchmark; `data/raw/dunelm_prices_2026-10.csv`) |
| Store floor area (descriptive only) | Small-format UK homeware shops: 2,800 sq ft (Oliver Bonas Glasgow Fort) to 4,500 sq ft (Dunelm Westfield London) |

The generator is in `src/generate_data.py` (fixed random seed, so the results are reproducible).

In [1]:
import subprocess, sys
result = subprocess.run([sys.executable, "src/generate_data.py"], capture_output=True, text=True)
print(result.stdout or result.stderr)

stores              13 rows
customers       70,000 rows
products           126 rows
orders         341,966 rows
order_lines    827,926 rows
returns         45,902 rows



In [2]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
orders = pd.read_csv("data/simulated/orders.csv", parse_dates=["order_datetime"], dtype={"promo_code": str})
lines  = pd.read_csv("data/simulated/order_lines.csv")
stores = pd.read_csv("data/simulated/stores.csv")
city   = pd.read_csv("data/clean/city_profile.csv")
lines["net_sales"] = lines["unit_price"] * lines["quantity"] * (1 - lines["discount_pct"])
order_value = lines.groupby("order_id")["net_sales"].sum()
orders = orders.join(order_value, on="order_id")
print(f"{len(orders):,} orders, {len(lines):,} order lines, revenue £{orders['net_sales'].sum()/1e6:.1f}m")

341,966 orders, 827,926 order lines, revenue £16.8m


### Check 1: does each store's revenue follow its city's real retail output?

In [3]:
store_rev = (orders[orders["channel"] == "Store"].merge(stores[["store_id", "city"]], on="store_id")
             .groupby("city")["net_sales"].sum().rename("store_revenue"))
check = city.set_index("city")[["retail_gva_gbp_m", "retail_gva_is_estimate"]].join(store_rev)
corr = np.corrcoef(np.log(check["retail_gva_gbp_m"]), np.log(check["store_revenue"]))[0, 1]
print(f"Correlation (log scale) between real retail GVA and simulated store revenue: {corr:.2f}")
check.sort_values("store_revenue", ascending=False).round(0)

Correlation (log scale) between real retail GVA and simulated store revenue: 0.99


,retail_gva_gbp_m,retail_gva_is_estimate,store_revenue
city,,,
London,21938.0,False,2418360.0
Manchester,5819.0,False,1110756.0
Birmingham,4673.0,False,986242.0
Leeds,3993.0,False,928116.0
Dublin,2885.0,True,892149.0
Belfast,3107.0,False,870541.0
Glasgow,2676.0,True,800956.0
Liverpool,2393.0,False,744088.0
Edinburgh,1936.0,False,739318.0


### Check 2: does the monthly online share match the ONS series?

In [4]:
sim = orders.assign(month=orders["order_datetime"].dt.to_period("M")).groupby("month")["channel"].apply(lambda s: (s == "Online").mean() * 100)
ons = pd.read_csv("data/raw/uk_internet_share_of_retail_J4MC.csv", header=None, names=["period", "value"])
ons = ons[ons["period"].str.match(r"^\d{4} [A-Z]{3}$", na=False)]
ons.index = pd.to_datetime(ons["period"], format="%Y %b").dt.to_period("M")
both = pd.DataFrame({"simulated_online_order_share": sim, "ONS_internet_share_of_sales": pd.to_numeric(ons["value"])}).dropna()
print(f"Mean absolute gap: {(both.iloc[:, 0] - both.iloc[:, 1]).abs().mean():.1f} percentage points")
both.tail(6).round(1)

Mean absolute gap: 0.4 percentage points


,simulated_online_order_share,ONS_internet_share_of_sales
2026-04,27.6,27.6
2026-05,28.8,27.8
2026-06,29.1,28.5
2026-07,28.0,27.5
2026-08,27.3,27.3
2026-09,29.0,28.3


### Deliberate data-quality problems (fixed in SQL script 03)

| Problem | Where | How many |
|---|---|---|
| City written inconsistently (`" Leeds "`, `LEEDS`, `Mcr`) | customers.home_city | ~6% of customers |
| Duplicated order lines | order_lines | 0.3% |
| Staff test lines with price £0 | order_lines | 25 |
| Guest checkouts with no customer ID | orders.customer_id | ~18% of store orders |

In [5]:
cust = pd.read_csv("data/simulated/customers.csv")
print("Distinct raw spellings of city:", cust["home_city"].nunique(), "-> real cities:", cust["home_city"].str.strip().str.lower().replace("mcr", "manchester").nunique())
print("Duplicated order lines:", lines.duplicated("order_line_id").sum())
print("Zero-price lines:", (lines["unit_price"] == 0).sum())
print("Store orders without customer:", f"{orders.loc[orders['channel']=='Store','customer_id'].isna().mean():.1%}")

Distinct raw spellings of city: 61 -> real cities: 12
Duplicated order lines: 2476
Zero-price lines: 25
Store orders without customer: 18.0%


### Check 3: do the simulated results sit inside industry benchmarks?

| Measure | Benchmark | Source |
|---|---|---|
| Gross margin | 52.4% | Dunelm (UK's largest homeware retailer), FY25 results |
| Share of revenue from the top 20% of customers | 50–73% | Sharp, Romaniuk & Graham (2019); Kim, Singh & Winer (2017) |
| Online return rate | below the 15–17% all-category UK average | IMRG |
| Online ÷ in-store return rate | 2.8 | Appriss Retail & Deloitte (2024) |
| Monthly pattern | ONS all-retailing, 2023–2025 | ONS JE4W |

In [6]:
products = pd.read_csv("data/simulated/products.csv")
returns  = pd.read_csv("data/simulated/returns.csv")
l = lines[lines["unit_price"] > 0].drop_duplicates("order_line_id").merge(products[["product_id", "unit_cost"]], on="product_id")
gross_margin = 1 - (l["quantity"] * l["unit_cost"]).sum() / l["net_sales"].sum()

last12 = orders[(orders["order_datetime"] >= "2025-10-01") & orders["customer_id"].notna()]
spend = last12.groupby("customer_id")["net_sales"].sum().sort_values(ascending=False)
top20_share = spend.head(int(len(spend) * 0.2)).sum() / spend.sum()

chan = l.merge(orders[["order_id", "channel"]], on="order_id").groupby("channel")["order_line_id"].nunique()
ret = returns.merge(l[["order_line_id", "order_id"]], on="order_line_id").merge(orders[["order_id", "channel"]], on="order_id")
ret_rate = ret.groupby("channel")["return_id"].nunique() / chan

pd.DataFrame({
    "simulated": [f"{gross_margin:.1%}", f"{top20_share:.0%}", f"{ret_rate['Online']:.1%}", f"{ret_rate['Online'] / ret_rate['Store']:.1f}"],
    "benchmark": ["52.4%", "50–73%", "below 15–17%", "2.8"]},
    index=["Gross margin", "Top 20% of customers' share of revenue", "Online return rate (lines)", "Online ÷ store return rate"])

,simulated,benchmark
Gross margin,51.4%,52.4%
Top 20% of customers' share of revenue,58%,50–73%
Online return rate (lines),10.0%,below 15–17%
Online ÷ store return rate,2.9,2.8


### Check 4: is the number of in-store transactions plausible? (Fermi estimate)

The yearly order volume is an assumption, so it is checked from the other side: how many sales would one small shop make per day?
About 20 a day is roughly two an hour across an 11-hour trading day, which is believable for a 3,000–4,500 sq ft homeware shop. This is a plausibility check, not a benchmark: no public source gives transactions per store per day for UK homeware shops.


In [7]:
store_orders = orders[orders["channel"] == "Store"]
days = (orders["order_datetime"].max().normalize() - orders["order_datetime"].min().normalize()).days + 1
per_day = store_orders.groupby("store_id").size() / days
print(f"Days: {days:,} · store orders: {len(store_orders):,} · stores: {per_day.size}")
print(f"In-store transactions per store per day: {len(store_orders) / days / per_day.size:.1f} on average "
      f"(range {per_day.min():.1f} to {per_day.max():.1f}; London is the highest)")

Days: 1,004 · store orders: 248,069 · stores: 12
In-store transactions per store per day: 20.6 on average (range 14.3 to 45.5; London is the highest)
